# 01 — Generate and validate a smoke trajectory dataset

This notebook implements the first dataset-generation milestone in the development plan:

1. load the controlled F-16-vs-F-16 scenario configuration;
2. sample reproducible initial conditions;
3. run primitive scripted target behaviours;
4. create target-centric observable features and privileged forecasting labels;
5. write the canonical manifest/episode/trajectory Parquet layout; and
6. reload it and assert schema, action-space, chronology, and leakage invariants.

> **Scope:** BVR Sim is an external, pinned dependency and is not required to execute this notebook. The notebook uses a deliberately simple deterministic kinematic environment—not a flight-dynamics simulator—to exercise the repository's *real* generation pipeline. The final section identifies the one environment-factory seam to replace after validating the pinned BVR Sim entity observation and action mappings. Data produced by the stub is for integration testing only and must not be used to train or evaluate a flight model.


In [ ]:
from __future__ import annotations

import math
import os
from pathlib import Path

import pandas as pd
import yaml

from bvr_behavior_prediction.data.observable_columns import MODEL_FEATURE_COLUMNS
from bvr_behavior_prediction.data.privileged_columns import PRIVILEGED_COLUMNS
from bvr_behavior_prediction.data.schema import EPISODE_COLUMNS, validate_columns
from bvr_behavior_prediction.generation.dataset_builder import DatasetBuilder
from bvr_behavior_prediction.generation.episode_runner import EpisodeRunner
from bvr_behavior_prediction.generation.manifest import DatasetManifest
from bvr_behavior_prediction.generation.scenario_sampler import ScenarioSampler
from bvr_behavior_prediction.policies.kinematic import (
    AcceleratePolicy,
    ClimbPolicy,
    DeceleratePolicy,
    DescendPolicy,
    MaintainPolicy,
    TurnPolicy,
)
from bvr_behavior_prediction.simulator.bvr_adapter import BVRSimAdapter
from bvr_behavior_prediction.simulator.scenario_config import ScenarioConfig

# Work whether Jupyter starts in the repository root or in notebooks/.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "pyproject.toml").exists(), "Start Jupyter in this repository"

OUTPUT_DIR = Path(os.environ.get(
    "BVR_NOTEBOOK_OUTPUT",
    REPO_ROOT / "artifacts/datasets/bvr_f16_1v1_smoke_demo",
))
CONFIG_PATH = REPO_ROOT / "configs/scenarios/f16_1v1.yaml"
print(f"Repository: {REPO_ROOT}")
print(f"Output:     {OUTPUT_DIR}")


## 1. Freeze the experiment configuration

The committed YAML is the human-readable experiment input. `ScenarioConfig` validates the simulator-facing subset and supplies a stable hash for episode provenance. The sampling stage controls the initial-condition envelope separately.


In [ ]:
raw_config = yaml.safe_load(CONFIG_PATH.read_text())
sampling_stage = raw_config.pop("sampling_stage")
config = ScenarioConfig(**raw_config)

print(yaml.safe_dump(raw_config, sort_keys=False))
print("sampling_stage:", sampling_stage)
print("config_hash:   ", config.config_hash)


## 2. Sample reproducible initial conditions

Use explicit per-episode seeds. The smoke scale here is intentionally tiny so this notebook executes quickly; increase to the plan's 20–50 episodes per behaviour only after visual inspection and backend validation.


In [ ]:
import random

BASE_SEED = 20260908
EPISODES_PER_BEHAVIOUR = 2
sampler = ScenarioSampler(sampling_stage)

policies = [
    MaintainPolicy(),
    TurnPolicy("left"),
    TurnPolicy("right"),
    ClimbPolicy(),
    DescendPolicy(),
    AcceleratePolicy(),
    DeceleratePolicy(),
]
# Give commanded skills explicit labels; kinematic labels are derived separately later.
for policy, label in zip(
    policies,
    ("MAINTAIN", "TURN_LEFT", "TURN_RIGHT", "CLIMB", "DESCEND", "ACCELERATE", "DECELERATE"),
):
    policy.label = label

scenario_records = []
for policy_index, policy in enumerate(policies):
    for repeat in range(EPISODES_PER_BEHAVIOUR):
        seed = BASE_SEED + policy_index * 100 + repeat
        scenario = sampler.sample(random.Random(seed))
        scenario_records.append((seed, policy, scenario))

pd.DataFrame([
    {"seed": seed, "policy": policy.label, **scenario.__dict__}
    for seed, policy, scenario in scenario_records
]).head(10)


## 3. Provide a disposable integration-test environment

`EpisodeRunner` talks only to `BVRSimAdapter`, whose `env_factory` accepts the translated simulator configuration and returns a Gymnasium-like environment. The stub below implements that contract and turns the four native discrete action branches into simple motion. It is **not** BVR Sim and makes no aerodynamic claims.


In [ ]:
class NotebookKinematicEnv:
    """Minimal Gymnasium-like environment used only to test data plumbing."""

    def __init__(self, bvr_config: dict, scenario):
        self.config = bvr_config
        self.scenario = scenario
        self.dt = float(bvr_config["dt"])
        self.max_steps = int(bvr_config["max_steps"])
        self.steps = 0
        self.states = {}

    @staticmethod
    def _state(x, y, z, speed, heading):
        return {
            "x": x, "y": y, "z": z,
            "vx": speed * math.cos(heading),
            "vy": speed * math.sin(heading),
            "vz": 0.0,
            "heading": heading, "pitch": 0.0, "roll": 0.0, "speed": speed,
        }

    def reset(self, *, seed=None):
        separation_m = self.scenario.range_nm * 1852.0
        target_heading = math.radians(self.scenario.heading_difference_deg)
        self.states = {
            "observer": self._state(0.0, 0.0, 8000.0, 250.0, 0.0),
            "target": self._state(
                separation_m,
                self.scenario.lateral_offset_nm * 1852.0,
                8000.0 + self.scenario.altitude_difference_m,
                250.0 + self.scenario.speed_difference_ms,
                target_heading,
            ),
        }
        self.steps = 0
        return self.states, {"states": self.states}

    def _advance(self, state, action):
        heading_bin, altitude_bin, speed_bin, _fire = BVRSimAdapter.validate_action(action)
        state["heading"] += math.radians((heading_bin - 7) * 1.5) * self.dt
        state["speed"] = max(80.0, state["speed"] + (speed_bin - 4) * 1.0 * self.dt)
        climb_rate = (altitude_bin - 7) * 2.0
        state["vx"] = state["speed"] * math.cos(state["heading"])
        state["vy"] = state["speed"] * math.sin(state["heading"])
        state["vz"] = climb_rate
        state["x"] += state["vx"] * self.dt
        state["y"] += state["vy"] * self.dt
        state["z"] += state["vz"] * self.dt

    def step(self, actions):
        for role, action in zip(("observer", "target"), actions):
            self._advance(self.states[role], action)
        self.steps += 1
        truncated = self.steps >= self.max_steps
        return self.states, 0.0, False, truncated, {"states": self.states}

    def close(self):
        pass


## 4. Generate trajectories and episode provenance

For a quick smoke run, cap episodes at 40 decisions (16 seconds at 0.4 s). Each run gets a fresh environment so sampled initial conditions cannot leak across episodes. `EpisodeRunner` adds relative features, causal temporal derivatives, native target actions, and transition labels.


In [ ]:
smoke_config = ScenarioConfig(
    observer_aircraft=config.observer_aircraft,
    target_aircraft=config.target_aircraft,
    backend=config.backend,
    dt=config.dt,
    max_steps=40,
    weapons_enabled=config.weapons_enabled,
    observation_type=config.observation_type,
    sensor_mode=config.sensor_mode,
)

trajectory_rows = []
episode_rows = []
for episode_number, (seed, target_policy, scenario) in enumerate(scenario_records):
    episode_id = f"smoke-{episode_number:04d}"
    adapter = BVRSimAdapter(
        smoke_config,
        env_factory=lambda bvr_config, s=scenario: NotebookKinematicEnv(bvr_config, s),
    )
    rows = EpisodeRunner(adapter, MaintainPolicy(), target_policy).run(seed, episode_id)
    adapter.close()
    trajectory_rows.extend(rows)
    episode_rows.append({
        "episode_id": episode_id,
        "seed": seed,
        "bvr_sim_commit": "NOT_BVR_SIM_NOTEBOOK_STUB",
        "bvr_sim_backend": smoke_config.backend,
        "bvr_sim_config_hash": smoke_config.config_hash,
        "jsbsim_version": "not-used",
        "observer_aircraft_type": smoke_config.observer_aircraft,
        "target_aircraft_type": smoke_config.target_aircraft,
        "observer_model_version": "notebook-stub-v1",
        "target_model_version": "notebook-stub-v1",
        "initial_range_nm": scenario.range_nm,
        "initial_altitude_difference_m": scenario.altitude_difference_m,
        "initial_heading_difference_deg": scenario.heading_difference_deg,
        "initial_speed_difference_ms": scenario.speed_difference_ms,
        "initial_aspect_bin": "sampled",
        "scenario_bin": scenario.scenario_bin,
        "observer_policy": "MAINTAIN",
        "target_policy": target_policy.label,
        "weapons_enabled": smoke_config.weapons_enabled,
        "sensor_mode": smoke_config.sensor_mode,
        "episode_duration_s": rows[-1]["time_s"] if rows else 0.0,
        "termination_reason": "time_limit",
    })

trajectories = pd.DataFrame(trajectory_rows)
episodes = pd.DataFrame(episode_rows)
print(f"{len(episodes)} episodes, {len(trajectories)} trajectory rows")
trajectories[["episode_id", "time_s", "range", "target_skill",
              "target_action_heading_bin", "target_action_altitude_bin",
              "target_action_speed_bin"]].head()


## 5. Validate before writing

These checks deliberately happen before persistence. They catch missing canonical fields, malformed native actions, non-causal chronology, invalid feature values, and accidental overlap between the model-feature allow-list and privileged labels.


In [ ]:
validate_columns(trajectories.columns)
assert set(EPISODE_COLUMNS).issubset(episodes.columns)
assert not (set(MODEL_FEATURE_COLUMNS) & set(PRIVILEGED_COLUMNS))
assert trajectories[list(MODEL_FEATURE_COLUMNS)].notna().all().all()
assert trajectories.groupby("episode_id")["time_s"].apply(lambda s: s.is_monotonic_increasing).all()
assert trajectories["target_action_heading_bin"].between(0, 14).all()
assert trajectories["target_action_altitude_bin"].between(0, 14).all()
assert trajectories["target_action_speed_bin"].between(0, 8).all()
assert trajectories["target_action_fire"].between(0, 1).all()
print("All pre-write schema, chronology, feature, leakage, and action checks passed.")


## 6. Write the canonical dataset layout

The manifest records that this build came from the notebook stub so it cannot be mistaken for simulator-generated training data. A real build must replace the sentinel commit/model versions with inspected, pinned values.


In [ ]:
manifest = DatasetManifest(
    dataset_id="bvr_f16_1v1_smoke_demo",
    simulator={
        "name": "notebook-kinematic-stub",
        "commit": "NOT_BVR_SIM_NOTEBOOK_STUB",
        "backend": smoke_config.backend,
        "config_hash": smoke_config.config_hash,
    },
    aircraft={
        "observer": smoke_config.observer_aircraft,
        "target": smoke_config.target_aircraft,
        "model_version": "notebook-stub-v1",
    },
    generation={
        "purpose": "pipeline integration only; not model training",
        "base_seed": BASE_SEED,
        "episodes_per_behaviour": EPISODES_PER_BEHAVIOUR,
        "sampling_stage": sampling_stage,
        "policy_labels": [policy.label for policy in policies],
    },
    fdm={"type": "deterministic-notebook-stub"},
    observation={"type": smoke_config.observation_type, "sensor_mode": smoke_config.sensor_mode},
    weapons={"enabled": smoke_config.weapons_enabled},
)

DatasetBuilder(OUTPUT_DIR, manifest, shard_size=5).write(trajectory_rows, episode_rows)
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(OUTPUT_DIR))


## 7. Reload independently and inspect coverage

Reading the persisted files—not the in-memory frames—verifies the dataset can be consumed without BVR Sim. This is also the right point for plots and label-distribution reviews before scaling generation.


In [ ]:
stored_episodes = pd.read_parquet(OUTPUT_DIR / "episodes.parquet")
stored_trajectories = pd.concat(
    [pd.read_parquet(path) for path in sorted((OUTPUT_DIR / "trajectories").glob("*.parquet"))],
    ignore_index=True,
)
validate_columns(stored_trajectories.columns)
assert len(stored_episodes) == len(episodes)
assert len(stored_trajectories) == len(trajectories)

coverage = stored_trajectories.groupby("target_skill").agg(
    episodes=("episode_id", "nunique"),
    rows=("step", "size"),
    mean_range_m=("range", "mean"),
    mean_turn_rate=("target_turn_rate", "mean"),
    mean_climb_rate=("target_climb_rate", "mean"),
    mean_acceleration=("target_acceleration", "mean"),
)
coverage.round(3)


## 8. Connect the pinned BVR Sim backend

Keep all sampling, metadata, checking, and writing code above. Replace only the adapter factory:

```python
adapter = BVRSimAdapter(smoke_config, env_factory=make_real_bvr_env)
```

Implement `make_real_bvr_env(bvr_config)` against the **installed pinned revision** after inspecting its environment constructor and entity-observation ordering. The returned object must expose:

- `reset(seed=seed) -> (observation, info)`;
- `step((observer_action, target_action)) -> (observation, reward, terminated, truncated, info)`;
- canonical state dictionaries at `info["states"]["observer"]` and `info["states"]["target"]`; and
- `close()`.

Before accepting real data, verify that action tuples correspond exactly to `MultiDiscrete([15, 15, 9, 2])`, compare Python/C++ backend traces, replace every notebook-stub provenance sentinel, inspect several target-relative trajectories, and repeat the leakage assertions. Do not silently combine stub, Python-backend, and C++-backend episodes in one dataset version.
